# Fase 0 — Auditoria, qualidade e semântica do alvo radar

Este notebook **não executa a varredura pesada do Zarr**. A varredura é realizada pelo script `scripts/00_audit_dataset.py`, que lê o dataset em lotes e grava resultados reduzidos em `analysis_outputs/00_quality/`.

Nesta versão, a Fase 0 também audita explicitamente o caminho **PNG do radar → legenda RGB → cache `.npy` → patch → `target` do Zarr**. Quando o diretório do cache é informado, o script compara amostras do Zarr diretamente contra os `.npy` correspondentes.

O objetivo é responder, antes de qualquer análise meteorológica ou treinamento:

- a estrutura do Zarr está consistente?
- a ordem e o número de canais estão corretos?
- qual é a cobertura temporal real entre 2011 e 2024?
- qual é a geometria espacial efetiva dos patches?
- há NaN/Inf inesperado nas entradas?
- como o cache do radar é representado numericamente?
- o `target` armazenado realmente reproduz `log1p(clip(cache, 0, +∞))`?
- a máscara armazenada reproduz `isfinite(cache_patch)`?
- a unidade física do produto radar está declarada pelo builder ou ainda precisa ser confirmada na fonte original?

> **Regra científica:** o builder chama o campo do cache de `reflectivity`, mas não declara a unidade física como `dBZ`. Portanto, este notebook separa o que é comprovado pelo código do que ainda depende da documentação original do produto radar.


## 0. Execução do auditor

A execução completa recomendada, a partir da raiz do projeto, é:

```bash
export CORRDIFF_RADAR_CACHE_DIR=/caminho/para/o/cache_do_radar

python scripts/00_audit_dataset.py \
  --dataset-dir datasets/corrdiff_2011_2024 \
  --radar-cache-dir "$CORRDIFF_RADAR_CACHE_DIR" \
  --output-dir analysis_outputs/00_quality \
  --batch-samples 512 \
  --quantile-sample-values 200000 \
  --radar-cache-file-sample 128 \
  --radar-target-validation-samples 256 \
  --overwrite
```

Também é possível definir somente `CORRDIFF_RADAR_CACHE_DIR`; o script usa essa variável quando `--radar-cache-dir` não é informado.

Para validar rapidamente caminhos e estrutura sem percorrer todos os pixels do Zarr:

```bash
python scripts/00_audit_dataset.py \
  --dataset-dir datasets/corrdiff_2011_2024 \
  --radar-cache-dir "$CORRDIFF_RADAR_CACHE_DIR" \
  --output-dir analysis_outputs/00_quality_quick \
  --radar-cache-file-sample 32 \
  --radar-target-validation-samples 64 \
  --quick \
  --overwrite
```

A execução `full` calcula contagens de NaN/Inf, média, desvio padrão, mínimo e máximo de forma exata sobre o Zarr. Os quantis são aproximados por amostragem controlada. A auditoria do cache não enumera todos os arquivos de 2 minutos: ela amostra arquivos associados a timestamps efetivamente representados no dataset, evitando um `glob` de milhões de arquivos.


In [ ]:
from pathlib import Path
import json
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 200)

# Se o JupyterLab foi iniciado na raiz do projeto, os defaults abaixo funcionam.
# Você também pode definir CORRDIFF_DATASET_DIR e CORRDIFF_QUALITY_DIR no shell.
DATASET_DIR = Path(os.environ.get("CORRDIFF_DATASET_DIR", "datasets/corrdiff_2011_2024"))
QUALITY_DIR = Path(os.environ.get("CORRDIFF_QUALITY_DIR", "analysis_outputs/00_quality"))

print("DATASET_DIR:", DATASET_DIR.resolve())
print("QUALITY_DIR:", QUALITY_DIR.resolve())

In [ ]:
required_files = [
    QUALITY_DIR / "dataset_summary.json",
    QUALITY_DIR / "audit_warnings.json",
    QUALITY_DIR / "radar_target_semantics.json",
    QUALITY_DIR / "radar_legend_mapping.parquet",
    QUALITY_DIR / "yearly_coverage.parquet",
    QUALITY_DIR / "monthly_coverage.parquet",
    QUALITY_DIR / "hourly_coverage.parquet",
    QUALITY_DIR / "temporal_coverage.parquet",
]

missing = [str(path) for path in required_files if not path.exists()]
if missing:
    raise FileNotFoundError(
        "Execute scripts/00_audit_dataset.py antes deste notebook. Arquivos ausentes:\n- "
        + "\n- ".join(missing)
    )

with (QUALITY_DIR / "dataset_summary.json").open("r", encoding="utf-8") as f:
    summary = json.load(f)

with (QUALITY_DIR / "audit_warnings.json").open("r", encoding="utf-8") as f:
    audit_messages = json.load(f)

with (QUALITY_DIR / "radar_target_semantics.json").open("r", encoding="utf-8") as f:
    radar_semantics = json.load(f)

yearly = pd.read_parquet(QUALITY_DIR / "yearly_coverage.parquet")
monthly = pd.read_parquet(QUALITY_DIR / "monthly_coverage.parquet")
hourly = pd.read_parquet(QUALITY_DIR / "hourly_coverage.parquet")
temporal = pd.read_parquet(QUALITY_DIR / "temporal_coverage.parquet")
radar_legend = pd.read_parquet(QUALITY_DIR / "radar_legend_mapping.parquet")

channel_path = QUALITY_DIR / "channel_summary.parquet"
missing_path = QUALITY_DIR / "missing_data.parquet"
patch_position_path = QUALITY_DIR / "patch_position_counts.parquet"
radar_cache_stats_path = QUALITY_DIR / "radar_cache_sample_statistics.parquet"
radar_mapping_path = QUALITY_DIR / "radar_target_mapping_validation.parquet"

channel_summary = pd.read_parquet(channel_path) if channel_path.exists() else pd.DataFrame()
missing_data = pd.read_parquet(missing_path) if missing_path.exists() else pd.DataFrame()
patch_positions = pd.read_parquet(patch_position_path) if patch_position_path.exists() else pd.DataFrame()
radar_cache_stats = pd.read_parquet(radar_cache_stats_path) if radar_cache_stats_path.exists() else pd.DataFrame()
radar_mapping = pd.read_parquet(radar_mapping_path) if radar_mapping_path.exists() else pd.DataFrame()

print("Status do audit:", summary.get("status"))
print("Modo:", summary.get("audit_mode"))
print("Tempo de execução (s):", round(summary.get("elapsed_seconds", 0), 2))
print("Cache radar:", summary.get("paths", {}).get("radar_cache_dir"))


## 1. Resumo estrutural


In [ ]:
dataset = summary["dataset"]
config = summary["configuration"]
temporal_summary = summary["temporal"]
patch_geometry = summary["patch_geometry"]

structural = pd.DataFrame(
    [
        ("Amostras (patches)", dataset.get("num_samples")),
        ("Canais de entrada", dataset.get("num_channels")),
        ("Shape input", str(dataset.get("input_shape"))),
        ("Shape target", str(dataset.get("target_shape"))),
        ("Shape mask", str(dataset.get("mask_shape"))),
        ("Dimensão amostral consistente", dataset.get("sample_dimension_consistent")),
        ("Transformação do target", dataset.get("target_transform")),
        ("Período configurado", f"{config.get('start_date')} → {config.get('end_date')}"),
        ("Frequência temporal", config.get("time_frequency")),
        ("Grade do radar", str(config.get("radar_grid_shape"))),
        ("Resolução do radar (km)", config.get("radar_resolution_km")),
        ("Patch size", config.get("patch_size")),
        ("Stride", config.get("stride")),
    ],
    columns=["Item", "Valor"],
)

display(structural)

In [ ]:
channels = pd.DataFrame(
    {
        "índice": range(len(dataset.get("channels", []))),
        "canal": dataset.get("channels", []),
    }
)
display(channels)

### Configuração esperada neste experimento

A configuração científica planejada é:

- **superfície:** `tcwv`, `t2m`, `u10`, `v10`;
- **850 hPa:** `t_850`, `r_850`, `u_850`, `v_850`;
- **500 hPa:** `t_500`, `r_500`, `u_500`, `v_500`;
- **12 canais ERA5** ao todo;
- **radar:** grade 70×84, resolução espacial de 2 km;
- **dataset:** frequência horária;
- **patches:** 32×32, stride 16.

A tabela acima é a fonte efetiva do Zarr/metadata e deve prevalecer sobre qualquer expectativa teórica.


## 2. Avisos e notas do auditor


In [ ]:
warnings = audit_messages.get("warnings", [])
notes = audit_messages.get("notes", [])

if warnings:
    display(Markdown("### ⚠️ Avisos"))
    for item in warnings:
        display(Markdown(f"- **{item['code']}** — {item['message']}"))
else:
    display(Markdown("✅ **Nenhum aviso estrutural crítico foi produzido pelo auditor.**"))

if notes:
    display(Markdown("### Notas"))
    for item in notes:
        display(Markdown(f"- **{item['code']}** — {item['message']}"))

## 3. Cobertura temporal global


In [ ]:
temporal_table = pd.DataFrame(
    [
        ("Timestamps esperados", temporal_summary.get("expected_timestamps")),
        ("Timestamps com pelo menos um patch", temporal_summary.get("available_timestamps")),
        ("Timestamps sem patch", temporal_summary.get("missing_timestamps")),
        ("Cobertura temporal", temporal_summary.get("coverage_ratio")),
        ("Primeiro timestamp observado", temporal_summary.get("first_observed_timestamp")),
        ("Último timestamp observado", temporal_summary.get("last_observed_timestamp")),
        ("Unidade detectada no Zarr", temporal_summary.get("timestamp_storage_unit_detected")),
    ],
    columns=["Métrica", "Valor"],
)
display(temporal_table)

coverage = temporal_summary.get("coverage_ratio")
if coverage is not None:
    print(f"Cobertura global: {coverage:.2%}")

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4.5))
ax.bar(yearly["year"].astype(str), yearly["coverage_ratio"] * 100)
ax.set_ylim(0, 100)
ax.set_ylabel("Cobertura temporal (%)")
ax.set_xlabel("Ano")
ax.set_title("Cobertura temporal do dataset por ano")
ax.grid(axis="y", alpha=0.25)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

display(yearly)

**Interpretação:** quedas localizadas em um ano não devem ser interpretadas inicialmente como sinal meteorológico. Primeiro devem ser confrontadas com os contadores do builder (`missing_radar`, `missing_era5`, patches rejeitados) e com a disponibilidade do arquivo histórico do radar.


## 4. Cobertura mensal — procura por lacunas sistemáticas


In [ ]:
coverage_matrix = monthly.pivot(index="year", columns="month", values="coverage_ratio") * 100
coverage_matrix = coverage_matrix.reindex(columns=range(1, 13))

fig, ax = plt.subplots(figsize=(12, 6))
im = ax.imshow(coverage_matrix.values, aspect="auto", vmin=0, vmax=100)
ax.set_title("Cobertura temporal por ano e mês (%)")
ax.set_xlabel("Mês")
ax.set_ylabel("Ano")
ax.set_xticks(range(12), range(1, 13))
ax.set_yticks(range(len(coverage_matrix.index)), coverage_matrix.index)
fig.colorbar(im, ax=ax, label="Cobertura (%)")
plt.tight_layout()
plt.show()

In [ ]:
worst_months = monthly.sort_values(["coverage_ratio", "year", "month"]).head(24)
display(worst_months)

## 5. Cobertura por hora UTC


In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.5))
ax.bar(hourly["hour_utc"], hourly["coverage_ratio"] * 100)
ax.set_xticks(range(24))
ax.set_ylim(0, 100)
ax.set_xlabel("Hora UTC")
ax.set_ylabel("Cobertura (%)")
ax.set_title("Cobertura de timestamps por hora UTC")
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

display(hourly)

**Por que isso importa:** uma cobertura muito diferente entre horas pode enviesar a análise posterior do ciclo diurno. Antes de concluir que determinada hora possui mais ou menos precipitação, precisamos saber se ela possui a mesma disponibilidade observacional.


## 6. Quantidade de patches por timestamp


In [ ]:
patch_stats = temporal_summary.get("patches_per_timestamp", {})
display(pd.DataFrame([patch_stats]))

available_temporal = temporal.loc[temporal["available"]].copy()
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.hist(available_temporal["samples"], bins=np.arange(0.5, available_temporal["samples"].max() + 1.5, 1))
ax.set_xlabel("Patches gravados no timestamp")
ax.set_ylabel("Número de timestamps")
ax.set_title("Distribuição de patches válidos por timestamp")
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

Para a grade 70×84 com `patch_size=32` e `stride=16`, a geometria convencional do builder produz no máximo **12 âncoras** por campo completo (3 posições verticais × 4 horizontais), antes dos filtros de validade.


## 7. Geometria espacial dos patches


In [ ]:
geometry_table = pd.DataFrame(
    [
        ("Âncoras esperadas por campo", patch_geometry.get("expected_patch_positions_per_full_field")),
        ("Âncoras observadas", patch_geometry.get("actual_unique_patch_positions")),
        ("Cobertura espacial esperada", patch_geometry.get("expected_spatial_coverage_ratio")),
        ("Cobertura espacial observada", patch_geometry.get("actual_spatial_coverage_ratio")),
    ],
    columns=["Métrica", "Valor"],
)
display(geometry_table)

if not patch_positions.empty:
    pivot = patch_positions.pivot(index="patch_row", columns="patch_col", values="samples").fillna(0)
    fig, ax = plt.subplots(figsize=(8, 5))
    im = ax.imshow(pivot.values, aspect="auto")
    ax.set_xticks(range(len(pivot.columns)), pivot.columns)
    ax.set_yticks(range(len(pivot.index)), pivot.index)
    ax.set_xlabel("patch_col")
    ax.set_ylabel("patch_row")
    ax.set_title("Número de amostras por posição de patch")
    fig.colorbar(im, ax=ax, label="Amostras")
    plt.tight_layout()
    plt.show()
    display(patch_positions)

In [ ]:
coverage_file = QUALITY_DIR / "expected_patch_spatial_coverage.npy"
if coverage_file.exists():
    patch_coverage = np.load(coverage_file)
    fig, ax = plt.subplots(figsize=(9, 6))
    im = ax.imshow(patch_coverage, origin="upper")
    ax.set_title("Número de patches que cobrem cada pixel da grade 70×84")
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    fig.colorbar(im, ax=ax, label="Número de sobreposições")
    plt.tight_layout()
    plt.show()

    uncovered = np.argwhere(patch_coverage == 0)
    print("Pixels sem cobertura de patch:", len(uncovered))
    print("Cobertura espacial:", f"{(patch_coverage > 0).mean():.2%}")

### Observação metodológica sobre as bordas

Com o loop atual baseado em `range(0, tamanho - patch_size + 1, stride)`, uma dimensão cujo restante não seja compatível com o stride pode deixar pixels de borda sem cobertura. Isso **não invalida** o dataset, mas precisa ser conhecido ao reconstruir campos completos ou calcular climatologias espaciais a partir dos patches.

Uma alternativa futura seria adicionar explicitamente a última âncora `tamanho - patch_size` para garantir cobertura integral, caso isso seja desejável no desenho experimental.


## 8. Qualidade numérica dos canais


In [ ]:
if channel_summary.empty:
    display(Markdown("O auditor foi executado em modo `quick` sem estatísticas numéricas completas."))
else:
    cols = [
        c for c in [
            "kind", "channel_index", "channel", "finite_ratio", "mean", "std", "min", "max",
            "p01", "p05", "p25", "p50", "p75", "p95", "p99",
            "quantile_sample_count", "mean_abs_diff_vs_builder", "std_abs_diff_vs_builder"
        ] if c in channel_summary.columns
    ]
    display(channel_summary[cols])

In [ ]:
if not missing_data.empty:
    display(missing_data)

    nonfinite = missing_data.copy()
    nonfinite["nonfinite_count"] = (
        nonfinite["nan_count"] + nonfinite["posinf_count"] + nonfinite["neginf_count"]
    )
    fig, ax = plt.subplots(figsize=(11, 5))
    ax.barh(nonfinite["channel"], nonfinite["nonfinite_count"])
    ax.set_xlabel("NaN + Inf")
    ax.set_title("Valores não finitos por canal/array")
    ax.grid(axis="x", alpha=0.25)
    plt.tight_layout()
    plt.show()

No dataset atual, `minimum_input_valid_ratio=1.0` implica que patches com qualquer insuficiência de validade no tensor de entrada deveriam ser rejeitados antes da escrita. Portanto, encontrar NaN/Inf no `input` gravado merece investigação.


## 9. Máscara e integridade do target


In [ ]:
mask_info = summary.get("mask", {})
target_info = summary.get("target_integrity", {})

display(pd.DataFrame([mask_info]))
display(pd.DataFrame([target_info]))

if mask_info:
    print("Proporção global de pixels válidos do radar:", f"{mask_info.get('valid_pixel_ratio', float('nan')):.2%}")

### Transformação do alvo registrada no Zarr

O builder grava no atributo/metadata `target_transform = "log1p"`. A seção seguinte verifica de forma independente **o que existe no cache** e, quando o cache está acessível, reproduz numericamente a transformação cache → `target` para uma amostra de patches.


## 10. Semântica do radar: PNG → cache `.npy` → `target` Zarr

A implementação auditada do builder possui uma etapa própria de criação do cache do Radar Sumaré. O código:

1. lê o PNG e converte a imagem para RGB;
2. compara cada pixel às cores configuradas na legenda;
3. escolhe as duas cores de legenda mais próximas em distância Euclidiana RGB;
4. interpola numericamente entre os valores associados a essas duas cores;
5. remapeia o campo para a grade geográfica do radar por índices inteiros `py, px`;
6. grava a grade resultante como `float32` em `YYYYMMDD_HH_MM.npy`;
7. durante a criação dos patches, calcula `mask = isfinite(cache_patch)`;
8. aplica `nan_to_num`, `clip(min=0)` e `log1p` antes de gravar `target`.

**Importante:** o código nomeia o campo como `reflectivity`, porém **não declara a unidade física como dBZ**. O valor físico da legenda deve ser confirmado na documentação/origem do produto radar antes de usar o rótulo `dBZ` em gráficos ou na dissertação.


In [ ]:
builder_semantics = radar_semantics.get("builder_semantics", {})
cache_validation = radar_semantics.get("cache_validation", {})
mapping_validation = radar_semantics.get("target_mapping_validation", {})

display(Markdown("### Semântica documentada pelo código do builder"))
display(pd.DataFrame([
    ("Origem", builder_semantics.get("source_format")),
    ("Conversão RGB", builder_semantics.get("rgb_to_numeric_method")),
    ("Campo nomeado no código", builder_semantics.get("physical_field_name_in_builder")),
    ("Unidade física declarada", builder_semantics.get("physical_unit_declared_in_builder")),
    ("dBZ confirmado somente pelo builder?", builder_semantics.get("dbz_confirmed_by_builder_alone")),
    ("dtype do cache", builder_semantics.get("cache_array_dtype")),
    ("Transformação do target", builder_semantics.get("zarr_target_transform")),
    ("Definição da máscara", builder_semantics.get("zarr_mask_definition")),
], columns=["Item", "Valor"]))

display(Markdown("### Legenda RGB incorporada no builder"))
legend_display = radar_legend.copy()
legend_display["RGB"] = legend_display.apply(lambda r: f"({int(r.r)}, {int(r.g)}, {int(r.b)})", axis=1)
display(legend_display[["legend_index", "RGB", "value"]])


In [ ]:
# Relação determinística entre os valores da legenda e o target, caso o cache
# contenha exatamente esses valores e eles sejam não-negativos.
legend_transform = radar_legend[["value"]].copy().sort_values("value")
legend_transform["target_log1p"] = np.log1p(np.clip(legend_transform["value"].astype(float), 0, None))
legend_transform["expm1_target"] = np.expm1(legend_transform["target_log1p"])
display(legend_transform)

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(legend_transform["value"], legend_transform["target_log1p"], marker="o")
ax.set_xlabel("Valor numérico da legenda/cache")
ax.set_ylabel("Valor armazenado após log1p")
ax.set_title("Compressão numérica introduzida por log1p")
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()


### 10.1 Estatísticas amostradas diretamente do cache

O auditor **não percorre todos os arquivos de 2 minutos**. Ele escolhe timestamps distintos que aparecem no Zarr e inspeciona os `.npy` correspondentes. Isso permite verificar `dtype`, shape, faixa numérica, valores negativos/não finitos e possíveis valores fora da faixa da legenda sem enumerar milhões de arquivos.


In [ ]:
display(pd.DataFrame([cache_validation]))

if radar_cache_stats.empty:
    display(Markdown(
        "⚠️ **A inspeção direta do cache não foi executada.** Informe `--radar-cache-dir` "
        "ou defina `CORRDIFF_RADAR_CACHE_DIR` e rode novamente o auditor."
    ))
else:
    cols = [c for c in [
        "timestamp", "exists", "shape", "dtype", "finite_ratio",
        "min", "p01", "p50", "p95", "p99", "max",
        "negative_count", "below_legend_min_count", "above_legend_max_count"
    ] if c in radar_cache_stats.columns]
    display(radar_cache_stats[cols].head(30))

    present = radar_cache_stats.loc[radar_cache_stats.get("exists", False) == True].copy()
    if not present.empty and {"min", "max"}.issubset(present.columns):
        fig, ax = plt.subplots(figsize=(11, 4.5))
        x = np.arange(len(present))
        ax.scatter(x, present["min"], s=12, label="mínimo")
        ax.scatter(x, present["max"], s=12, label="máximo")
        ax.axhline(radar_legend["value"].min(), linestyle="--", linewidth=1, label="mín. legenda")
        ax.axhline(radar_legend["value"].max(), linestyle="--", linewidth=1, label="máx. legenda")
        ax.set_xlabel("Arquivo de cache amostrado")
        ax.set_ylabel("Valor numérico")
        ax.set_title("Faixa numérica observada nos arquivos de cache amostrados")
        ax.legend()
        ax.grid(alpha=0.2)
        plt.tight_layout()
        plt.show()


### 10.2 Validação direta cache → patch → Zarr

Para amostras do Zarr, o auditor usa `timestamp`, `patch_row` e `patch_col` para abrir o `.npy` original e extrair exatamente a mesma região espacial. Em seguida reproduz:

```python
expected_mask = np.isfinite(cache_patch)
filled = np.nan_to_num(cache_patch, nan=0.0, posinf=0.0, neginf=0.0)
clipped = np.clip(filled, 0.0, None)
expected_target = np.log1p(clipped)
```

Ele compara `expected_target` com `train.zarr/target` e `expected_mask` com `train.zarr/mask`. Também testa `expm1(target)` contra o **cache após o clip**, deixando explícito que valores negativos, caso existam, não podem ser recuperados.


In [ ]:
display(pd.DataFrame([mapping_validation]))

if radar_mapping.empty:
    display(Markdown("⚠️ Não há resultados de validação direta cache → Zarr."))
else:
    cols = [c for c in [
        "sample_index", "timestamp", "patch_row", "patch_col",
        "raw_cache_min", "raw_cache_max", "raw_cache_negative_count",
        "stored_target_min", "stored_target_max",
        "max_abs_diff_stored_vs_expected_log1p",
        "max_abs_diff_expm1_vs_clipped_cache",
        "mask_mismatch_count", "mapping_matches_within_tolerance"
    ] if c in radar_mapping.columns]
    display(radar_mapping[cols].head(30))

    valid = radar_mapping.loc[radar_mapping.get("shape_match", True) == True].copy()
    if not valid.empty and "max_abs_diff_stored_vs_expected_log1p" in valid.columns:
        fig, ax = plt.subplots(figsize=(9, 4.5))
        ax.hist(valid["max_abs_diff_stored_vs_expected_log1p"].dropna(), bins=40)
        ax.set_xlabel("Erro absoluto máximo por patch")
        ax.set_ylabel("Número de patches")
        ax.set_title("Validação da transformação cache → log1p → Zarr")
        ax.grid(axis="y", alpha=0.25)
        plt.tight_layout()
        plt.show()


### 10.3 Conclusão semântica a registrar

Se `target_mapping_validation.status == PASS`, podemos afirmar a partir do **código + verificação empírica** que:

- o cache `.npy` contém valores numéricos derivados da legenda RGB do produto radar;
- esses valores são gravados em `float32` na grade do radar;
- o `target` do Zarr não contém o cache bruto;
- para valores finitos e não negativos, `target = log1p(cache)`;
- `expm1(target)` retorna o valor do cache **após o `clip(min=0)`**;
- valores negativos do cache, se existirem, são convertidos para zero e sua magnitude original é perdida;
- a máscara é definida pela finitude do cache **antes** do `clip`;
- **a unidade física (`dBZ` ou outra) não é declarada pelo builder** e deve ser confirmada na fonte do produto de radar.

Essa distinção é importante para a Fase 1: podemos analisar matematicamente a distribuição do `target`, mas só devemos rotular valores como `dBZ` depois de comprovar a unidade original da legenda.


## 11. Contadores do builder


In [ ]:
builder_counters = summary.get("builder_counters", {})
if builder_counters:
    counters_df = pd.DataFrame(
        sorted(builder_counters.items()), columns=["contador", "valor"]
    )
    display(counters_df)
else:
    print("metadata.json não contém counters do builder.")

## 12. Diagnóstico automático para avançar à Fase 1


In [ ]:
checks = []

def add_check(name, ok, detail):
    checks.append({"checagem": name, "status": "OK" if ok else "REVISAR", "detalhe": detail})

add_check(
    "Dimensões do Zarr",
    bool(dataset.get("sample_dimension_consistent")),
    str(dataset.get("sample_dimensions")),
)
add_check(
    "12 canais ERA5",
    dataset.get("num_channels") == 12,
    f"C={dataset.get('num_channels')} | {dataset.get('channels')}",
)
add_check(
    "Grade do radar 70×84",
    config.get("radar_grid_shape") == [70, 84],
    str(config.get("radar_grid_shape")),
)
add_check(
    "Patch 32 / stride 16",
    config.get("patch_size") == 32 and config.get("stride") == 16,
    f"patch={config.get('patch_size')}, stride={config.get('stride')}",
)
add_check(
    "Cobertura temporal conhecida",
    temporal_summary.get("coverage_ratio") is not None,
    f"coverage={temporal_summary.get('coverage_ratio')}",
)

if not missing_data.empty:
    input_rows = missing_data.loc[missing_data["kind"] == "input"]
    input_nonfinite = int(
        input_rows[["nan_count", "posinf_count", "neginf_count"]].sum().sum()
    )
    add_check("Inputs sem NaN/Inf", input_nonfinite == 0, f"nonfinite={input_nonfinite}")

if summary.get("mask"):
    add_check(
        "Máscara binária",
        summary["mask"].get("binary_violation_count", 0) == 0,
        f"violations={summary['mask'].get('binary_violation_count')}",
    )

mapping_status = radar_semantics.get("target_mapping_validation", {}).get("status")
if mapping_status not in (None, "SKIPPED"):
    add_check(
        "Cache → target reproduz o builder",
        mapping_status == "PASS",
        str(radar_semantics.get("target_mapping_validation", {})),
    )
else:
    add_check(
        "Cache → target foi validado diretamente",
        False,
        "SKIPPED — rode novamente com --radar-cache-dir para fechar a semântica do alvo.",
    )

add_check(
    "Unidade física do radar documentada",
    builder_semantics.get("physical_unit_declared_in_builder") is not None,
    "O builder não declara dBZ; confirmar na documentação/origem da legenda do produto radar.",
)

checks_df = pd.DataFrame(checks)
display(checks_df)

if (checks_df["status"] == "REVISAR").any() or summary.get("warnings_count", 0) > 0:
    display(Markdown("### Resultado: **há itens a revisar antes da Fase 1**"))
else:
    display(Markdown("### Resultado: **dataset apto a avançar para a Fase 1 — análise univariada**"))


## 13. O que levamos para a próxima fase

Ao concluir esta auditoria, registre no texto da dissertação:

1. período temporal efetivamente disponível;
2. número de timestamps e patches válidos;
3. cobertura por ano/mês/hora;
4. ordem exata dos 12 canais;
5. resolução espacial e temporal;
6. geometria e cobertura dos patches;
7. percentual de pixels válidos do radar;
8. presença/ausência de valores não finitos;
9. legenda numérica implementada na criação do cache;
10. faixa efetivamente observada nos `.npy` amostrados;
11. validação direta da transformação cache → `target`;
12. efeito do `clip(min=0)` e eventual perda de valores negativos;
13. situação da unidade física do radar — **não declarada no builder e pendente de confirmação externa**;
14. qualquer período ou posição espacial sub-representada.

A **Fase 1** deverá então estudar as distribuições univariadas em maior detalhe: histogramas, ECDF, quantis, assimetria, curtose e dependência sazonal. Para o radar, mantenha duas escalas explicitamente separadas quando necessário: `target` armazenado e `expm1(target)` (domínio pós-clip do cache). Não use o rótulo `dBZ` até a unidade original da legenda estar documentada.
